# theoryminer — installation test in Google Colab

This notebook checks that `theoryminer` installs from PyPI with pip and that each public function works.
Run the cells from top to bottom. Each cell ends with a check. A check that fails stops the cell with an
`AssertionError`. At the end, save the notebook, so that the outputs are stored in the file.

## 1. Install from PyPI

In [3]:
%pip install -q theoryminer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. The environment

In [4]:
import os
import sys
import torch
import theoryminer

package_dir = os.path.dirname(theoryminer.__file__)
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("theoryminer:", package_dir)
print("data files :", sorted(os.listdir(os.path.join(package_dir, "data"))))

assert os.path.isfile(os.path.join(package_dir, "data", "ELSST_R5.rdf.gz")), "the ELSST file is missing from the package"
print("check passed: the package holds the ELSST file")

python     : 3.13.15
torch      : 2.11.0+cpu | cuda available: False
theoryminer: /usr/local/lib/python3.13/dist-packages/theoryminer
data files : ['ELSST_LICENSE.md', 'ELSST_R5.rdf.gz']
check passed: the package holds the ELSST file


## 3. Download the sample papers

In [5]:
import json
import urllib.request

REPO = "rasoulnorouzi/theoryminer"


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    return listing


listing = download_sample_files()
print(len(listing), "files:", sorted(item["name"] for item in listing))
assert len(listing) == 8, "expected 7 PDF files and SOURCES.md"
print("check passed: 8 files downloaded")

8 files: ['SOURCES.md', 'alrabai_2021_autonomy_supportive_teaching.pdf', 'barrech_2018_job_insecurity_health.pdf', 'friesinger_2025_discrimination_health.pdf', 'gordesli_2024_social_media_mental_health.pdf', 'pelikan_2021_needs_intrinsic_motivation.pdf', 'schutz_2025_loneliness_social_support.pdf', 'stephany_2017_income_inequality_trust.pdf']
check passed: 8 files downloaded


## 4. `harvest()`: one PDF, a folder, and bad files

In [6]:
import shutil
from theoryminer import harvest

data = harvest("sample_files")
sent_ids = [s["sent_id"] for s in data["sentences"]]
doc_ids = {s["doc_id"] for s in data["sentences"]}

assert len(doc_ids) == 7, "expected 7 documents"
front = [s for s in data["sentences"] if s["flag"] == "front_matter"]
assert len(front) > 0, "expected some sidebar sentences with the flag front_matter"
assert len(sent_ids) == len(set(sent_ids)), "sent_id is not unique"
print()
print("check passed:", len(sent_ids), "sentences from", len(doc_ids), "documents; every sent_id is unique;",
      len(front), "flagged front_matter")

  reading PDF 7 of 7
harvest: sample_files
  documents read: 7 | files skipped: 1
  sentences kept: 2,547 | dropped rows: 1,850
       423  table_row
       328  no_lowercase
       247  reference_section
       229  too_short
       148  low_alpha
       107  running_head
       106  heading
        74  digit_heavy
        72  spaced_letters
        52  caption_or_keywords
        31  ref_fragment
        13  index_entry
         8  doi_url
         7  too_long
         2  empty_page
         1  not_pdf
         1  page_number
         1  front_back_matter
  sentences per document:
       265  alrabai_2021_autonomy_supportive_teaching
       297  barrech_2018_job_insecurity_health
       301  friesinger_2025_discrimination_health
       385  gordesli_2024_social_media_mental_health
       318  pelikan_2021_needs_intrinsic_motivation
       616  schutz_2025_loneliness_social_support
       365  stephany_2017_income_inequality_trust
  skipped: SOURCES.md: the file name does not end with

In [7]:
# A folder with one good PDF and three bad files.
os.makedirs("bad_folder", exist_ok=True)
shutil.copy("sample_files/barrech_2018_job_insecurity_health.pdf", "bad_folder/good.pdf")
with open("bad_folder/notes.txt", "w") as fh:
    fh.write("A text file, not a PDF.")
with open("bad_folder/webpage.pdf", "w") as fh:
    fh.write("<!DOCTYPE html><html><body>Access denied</body></html>")
with open("bad_folder/empty.pdf", "w") as fh:
    fh.write("")

print("bad_folder holds", len(os.listdir("bad_folder")), "files:", sorted(os.listdir("bad_folder")))
print()
mixed = harvest("bad_folder", cache=False)
skipped = {}
for row in mixed["dropped"]:
    if row["page"] == 0:
        skipped[row["doc_id"]] = row["drop_reason"]
print()
print("skipped files:", skipped)
assert skipped == {"notes": "not_pdf", "webpage": "unreadable_pdf", "empty": "unreadable_pdf"}
print("check passed: the bad files are logged, and the good file is read")

  reading PDF 3 of 3
harvest: bad_folder
  documents read: 1 | files skipped: 3
  sentences kept: 297 | dropped rows: 210
        94  no_lowercase
        47  table_row
        32  too_short
        10  spaced_letters
         8  digit_heavy
         5  heading
         4  caption_or_keywords
         4  low_alpha
         3  running_head
         2  unreadable_pdf
         1  not_pdf
  skipped: notes.txt: the file name does not end with .pdf
  skipped: empty.pdf: Not a PDF: file is empty
  skipped: webpage.pdf: Not a PDF: file appears to be HTML

skipped files: {'notes': 'not_pdf', 'empty': 'unreadable_pdf', 'webpage': 'unreadable_pdf'}
check passed: the bad files are logged, and the good file is read


## 4b. `extract_dois()`: the DOI of each paper

In [ ]:
from theoryminer import extract_dois

ANSWER_KEY = {  # read by hand from page 1 of each PDF
    "alrabai_2021_autonomy_supportive_teaching.pdf": "10.3389/fpsyg.2021.728657",
    "barrech_2018_job_insecurity_health.pdf": "10.1186/s12889-018-5621-4",
    "friesinger_2025_discrimination_health.pdf": "10.1186/s12889-025-23888-6",
    "gordesli_2024_social_media_mental_health.pdf": "10.1371/journal.pone.0316365",
    "pelikan_2021_needs_intrinsic_motivation.pdf": "10.1371/journal.pone.0257346",
    "schutz_2025_loneliness_social_support.pdf": "10.1186/s12889-025-23247-5",
    "stephany_2017_income_inequality_trust.pdf": "10.1007/s11205-016-1460-9",
}
rows = extract_dois("sample_files", save="outputs")
found = {}
for row in rows:
    if row["status"] == "found":
        found[row["file"]] = row["doi"]
assert found == ANSWER_KEY, "a DOI differs from the answer key"
assert os.path.isfile("outputs/dois.csv"), "the CSV file is missing"

bad = extract_dois("bad_folder", cache=False)
status = {}
for row in bad:
    status[row["file"]] = row["status"]
assert status == {"good.pdf": "found", "notes.txt": "not_pdf",
                  "webpage.pdf": "unreadable_pdf", "empty.pdf": "unreadable_pdf"}
print()
print("check passed: 7 of 7 DOIs are correct, the CSV file exists, and the bad files get a status")

## 5. `causenet()`: find the cause-effect pairs

In [8]:
from theoryminer import causenet

toy = causenet(["Smoking causes lung cancer and heart disease",
                "Exercise improves physical health and mental well-being",
                "This is a non-causal sentence about weather"])
print(toy[:2])
assert len(toy) >= 2, "the model found fewer relations than expected"
print("check passed: the model loads and finds relations")

causenet: loading rasoultilburg/SocioCausaNet ...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

configuration_joint_causal.py:   0%|          | 0.00/895 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/rasoultilburg/SocioCausaNet:
- configuration_joint_causal.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

causenet: device = cpu
  3/3 sentences (1s)
causenet: 2 causal sentences of 3 | 4 relations
  spans repaired after decoding: 0 | both spans found verbatim in their sentence: 100%
[{'rel_id': 's0000r00', 'sent_id': 's0000', 'doc_id': 'input', 'page': 0, 'cause': 'smoking', 'effect': 'lung cancer', 'sentence': 'Smoking causes lung cancer and heart disease'}, {'rel_id': 's0000r01', 'sent_id': 's0000', 'doc_id': 'input', 'page': 0, 'cause': 'smoking', 'effect': 'heart disease', 'sentence': 'Smoking causes lung cancer and heart disease'}]
check passed: the model loads and finds relations


In [9]:
relations = causenet(data["sentences"])
rel_ids = [r["rel_id"] for r in relations]
assert len(rel_ids) == len(set(rel_ids)), "rel_id is not unique"
print()
print("check passed:", len(relations), "relations; every rel_id is unique")
for r in relations[:5]:
    print(f"  [{r['rel_id']} p{r['page']}] {r['cause']!r} -> {r['effect']!r}")

  2,547/2,547 sentences (845s)
causenet: 481 causal sentences of 2,547 | 979 relations
  spans repaired after decoding: 124 | both spans found verbatim in their sentence: 97%

check passed: 979 relations; every rel_id is unique
  [d000p0003s003r00 p1] 'experimental intervention' -> 'efl ehsan namaziandost'
  [d000p0006s003r00 p1] 'desire' -> 'autonomy'
  [d000p0006s003r01 p1] 'desire' -> 'take charge'
  [d000p0006s003r02 p1] 'desire' -> 's own leaning'
  [d000p0006s003r03 p1] 'ability' -> 'autonomy'


## 6. The harmonizer: ELSST, grouping, clustering, names

In [10]:
from theoryminer.harmonizer import (
    standardize_constructs, standardize_groups, pairwise_grouping, clusterer, label_groups)

probe = ["loneliness", "social trust", "unemployment", "intrinsic motivation"]
matches = standardize_constructs(probe, top=3)
print()
for span in probe:
    best = matches[span][0]
    print(f"  {span:<22} -> {best['leaf']} ({best['score']:.2f})")
assert all(len(matches[span]) == 3 for span in probe)
print("check passed: the packaged ELSST file loads, and the spans get concepts")

taxonomy: 3,436 concepts | 2,417 leaves | 2,661 entries (leaves only)
taxonomy: no cache yet for (elsst, enriched, allmpnet, leaves). Embedding 2,661 entries now; about 2 min on CPU. This runs once; later calls load elsst_enriched_allmpnet_leaves.npy.
embeddings: loading sentence-transformers/all-mpnet-base-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

standardize: 4 spans -> top 3 of 2,661 entries | best score: median 0.65, min 0.63, max 0.69

  loneliness             -> LONELINESS (0.63)
  social trust           -> INTERPERSONAL TRUST (0.69)
  unemployment           -> UNEMPLOYED (0.64)
  intrinsic motivation   -> INCENTIVES (0.65)
check passed: the packaged ELSST file loads, and the spans get concepts


In [11]:
groups = pairwise_grouping(relations, threshold=0.69)
clusters = clusterer(relations)
clusters_umap = clusterer(relations, umap=dict(n_components=5, n_neighbors=15, min_dist=0.0))
named = standardize_groups(clusters, name_by="mean_sim", top=3)
labelled = label_groups(clusters, by="both", top=3)

spans = {r["cause"] for r in relations} | {r["effect"] for r in relations}
assert set(groups) == spans and set(clusters) == spans, "a span is missing from a result"
print()
print("check passed: every span has a group, a cluster and a name")

pairwise: 1,219 spans -> 669 groups at cos >= 0.69 | 90 groups with 2+ members, largest 153
cluster: 1,219 spans -> 244 clusters | 480 noise (39%) | raw embeddings


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster: 1,219 spans -> 287 clusters | 178 noise (15%) | umap {'n_components': 5, 'n_neighbors': 15, 'min_dist': 0.0}
taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)
standardize: 724 groups named by mean_sim | best score: median 0.48, min 0.16, max 0.77
labels: 724 groups labelled by both | 244 with 2 or more members

check passed: every span has a group, a cluster and a name


In [13]:
named

{'efl ehsan namaziandost': {'central': 'efl ehsan namaziandost',
  'size': 1,
  'members': ['efl ehsan namaziandost'],
  'matches': [{'id': 'https://elsst.cessda.eu/id/5/',
    'leaf': 'ELSST Thesaurus',
    'path': 'ELSST Thesaurus',
    'score': 0.2528},
   {'id': 'https://elsst.cessda.eu/id/5/7a160488-c9bb-4b52-bf80-9c1170679656',
    'leaf': 'NEO-NAZISM',
    'path': 'POLITICS > POLITICAL THEORY > POLITICAL IDEOLOGIES > NATIONAL SOCIALISM > NEO-NAZISM',
    'score': 0.2456},
   {'id': 'https://elsst.cessda.eu/id/5/0a08d14b-cf21-4ae2-b448-12a4d8208449',
    'leaf': 'EXAMINATIONS',
    'path': 'EXAMINATIONS',
    'score': 0.2431}]},
 'desire': {'central': 'desire',
  'size': 1,
  'members': ['desire'],
  'matches': [{'id': 'https://elsst.cessda.eu/id/5/d81ee447-fde7-4df6-931f-0e4f1f5bdcc7',
    'leaf': 'ASPIRATION',
    'path': 'MOTIVATION > ASPIRATION',
    'score': 0.4314},
   {'id': 'https://elsst.cessda.eu/id/5/5d1671fa-2940-4d42-b993-599b3ed84e76',
    'leaf': 'INCENTIVES',
    

## 7. Summary

In [12]:
print("sentences :", len(data["sentences"]))
print("relations :", len(relations))
print("spans     :", len(spans))
print("clusters  :", len({g["group"] for g in clusters.values() if g["group"] != -1}))
print("ALL CHECKS PASSED")

sentences : 2547
relations : 979
spans     : 1219
clusters  : 244
ALL CHECKS PASSED
